In [ ]:
#|default_exp datamanager

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

## DataManager

This notebook defines the `DataManager` class, which handles caching and data retrieval.

In [ ]:
#| export

from token_data.datasource import DataSource
from pathlib import Path
from typing import List
import polars as pl
from datetime import datetime, timedelta
import logging

In [ ]:
#| export

class DataManager:
    """Manages data fetching, caching, and refreshing."""

    def __init__(self, datasource: DataSource):
        self.datasource = datasource

    def get_data(self, token: str, start_date: str, end_date: str, data_type: str = 'spot', time_interval: str = '1h', refresh_n_units: int = 0) -> pl.DataFrame:
        """Reads data from a file, fetches new data if needed, and caches it."""
        file_name = f"{token}_{data_type}_{time_interval}.parquet"
        df = self.datasource.read_data(file_name)

        requested_start_date = datetime.fromisoformat(start_date.replace('Z', ''))
        requested_end_date = datetime.fromisoformat(end_date.replace('Z', ''))

        if df.is_empty():
            df = self._fetch_and_save(token, start_date, end_date, data_type, time_interval, file_name)
        else:
            last_date = df['datetime'].max()
            if requested_end_date > last_date:
                try:
                    df = self._fetch_and_save(token, last_date.isoformat(), end_date, data_type, time_interval, file_name, df)
                except pl.exceptions.ShapeError:
                    logging.warning(f"Shape mismatch for {file_name}. Backing up and refetching.")
                    self._backup_and_remove(file_name)
                    df = self._fetch_and_save(token, start_date, end_date, data_type, time_interval, file_name)
            
            if refresh_n_units > 0:
                refresh_start_date = datetime.now() - timedelta(hours=refresh_n_units)
                df = df.filter(pl.col('datetime') < refresh_start_date)
                df = self._fetch_and_save(token, df['datetime'].max().isoformat(), end_date, data_type, time_interval, file_name, df)

        return df.filter((pl.col('datetime') >= requested_start_date) & (pl.col('datetime') <= requested_end_date))
    
    def _fetch_and_save(self, token: str, start_date: str, end_date: str, data_type: str, time_interval: str, file_name: str, existing_df: pl.DataFrame = None) -> pl.DataFrame:
        if data_type == 'spot':
            new_df = self.datasource.get_spot_prices(token, start_date, end_date, time_interval)
        elif data_type == 'perp':
            new_df = self.datasource.get_perp_prices(token, start_date, end_date, time_interval)
        elif data_type == 'funding':
            new_df = self.datasource.get_funding_rates(token, start_date, end_date)
        elif data_type == 'combined':
            new_df = self.datasource.get_combined_data(token, start_date, end_date, time_interval)
        else:
            raise ValueError(f"Unsupported data type: {data_type}")
        
        if not new_df.is_empty():
            if existing_df is not None:
                df = existing_df.vstack(new_df)
            else:
                df = new_df
            self.datasource.save_data(df, file_name)
            return df
        return existing_df if existing_df is not None else pl.DataFrame()

    def _backup_and_remove(self, file_name: str):
        backup_folder = self.datasource.data_folder / 'backup'
        backup_folder.mkdir(exist_ok=True)
        
        file_path = self.datasource.data_folder / file_name
        if file_path.exists():
            timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
            backup_path = backup_folder / f"{file_path.stem}_{timestamp}{file_path.suffix}"
            file_path.rename(backup_path)

        backups = sorted(backup_folder.glob(f"{file_path.stem}_*"), key=lambda f: f.stat().st_mtime, reverse=True)
        for old_backup in backups[5:]:
            old_backup.unlink()

### Tests

In [ ]:
import unittest
from pathlib import Path
import shutil
from token_data.coinbase import CoinbaseDataSource

class TestDataManager(unittest.TestCase):
    
    def setUp(self):
        self.data_folder = Path('./test_data')
        self.data_folder.mkdir(exist_ok=True)
        self.coinbase = CoinbaseDataSource(self.data_folder)
        self.datamanager = DataManager(self.coinbase)

    def tearDown(self):
        shutil.rmtree(self.data_folder)

    def test_get_data(self):
        end_date = datetime.now()
        start_date = end_date - timedelta(days=1)
        df = self.datamanager.get_data('BTC-USD', start_date.isoformat(), end_date.isoformat(), data_type='spot')
        self.assertIsInstance(df, pl.DataFrame)
        self.assertGreater(len(df), 0)
        self.assertEqual(df.columns, ['datetime', 'open', 'high', 'low', 'close', 'volume', 'token'])
        
    def test_backup_and_refresh(self):
        # Create a dummy file with a different schema
        file_name = 'BTC-USD_spot_1h.parquet'
        dummy_df = pl.DataFrame({'a': [1], 'b': [2]})
        self.coinbase.save_data(dummy_df, file_name)
        
        end_date = datetime.now()
        start_date = end_date - timedelta(days=1)
        df = self.datamanager.get_data('BTC-USD', start_date.isoformat(), end_date.isoformat(), data_type='spot')
        
        self.assertIsInstance(df, pl.DataFrame)
        self.assertGreater(len(df), 0)
        self.assertEqual(df.columns, ['datetime', 'open', 'high', 'low', 'close', 'volume', 'token'])

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)